# RansomSense
## Notebook 3: Feature Engineering & Data Preprocessing

### Objective

Machine learning models require clean, structured, and numerical input data.

The purpose of this notebook is to transform the ransomware dataset into a format suitable for predictive modeling.

The following preprocessing steps are performed:

- Feature selection
- Feature engineering
- Encoding categorical variables
- Scaling numerical features
- Train-test split
- Saving preprocessing components

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import joblib

In [2]:
df = pd.read_csv("../data/processed/cleaned_ransomware_dataset.csv")

df.head()

,Attack_ID,Company_Name,Industry,Country,Attack_Date,Ransomware_Group,Attack_Vector,Ransom_Amount_USD,Payment_Status,Data_Exfiltration,Downtime_Days,Records_Affected,Estimated_Total_Loss_USD,Severity_Score
0,RA-100000,Mcdonald-Smith,Government,United Kingdom,2024-08-02,BlackCat,Drive-by Download,938912,Paid,No,2,2307213,1462487,3.55
1,RA-100001,Lopez and Sons,Transportation,Japan,2021-03-07,LockBit,Credential Theft,254957,Not Paid,No,6,1834168,752882,2.99
2,RA-100002,Anthony Group,Healthcare,United Kingdom,2022-03-28,Ryuk,Malicious Attachment,3773238,Negotiated,Yes,38,2333732,5480736,11.81
3,RA-100003,"Rodriguez, Romero and Mills",Finance,Australia,2022-03-17,Maze,Credential Theft,2828559,Negotiated,Yes,7,778108,3635324,4.66
4,RA-100004,Brown LLC,Technology,South Africa,2025-02-03,Hive,Malicious Attachment,3858935,Negotiated,Yes,35,1047217,5803128,10.16


# Dataset Overview

The processed dataset is loaded and inspected before preprocessing begins.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Attack_ID                 5000 non-null   str    
 1   Company_Name              5000 non-null   str    
 2   Industry                  5000 non-null   str    
 3   Country                   5000 non-null   str    
 4   Attack_Date               5000 non-null   str    
 5   Ransomware_Group          5000 non-null   str    
 6   Attack_Vector             5000 non-null   str    
 7   Ransom_Amount_USD         5000 non-null   int64  
 8   Payment_Status            5000 non-null   str    
 9   Data_Exfiltration         5000 non-null   str    
 10  Downtime_Days             5000 non-null   int64  
 11  Records_Affected          5000 non-null   int64  
 12  Estimated_Total_Loss_USD  5000 non-null   int64  
 13  Severity_Score            5000 non-null   float64
dtypes: float64(1), int6

# Feature Selection

To prevent data leakage, only features that would realistically be available at the time of prediction are selected.

Features such as `Estimated_Total_Loss_USD`, `Downtime_Days`, and `Payment_Status` are excluded because they represent outcomes of the ransomware attack rather than information known beforehand.

The selected features are:

- Industry
- Country
- Attack Date
- Ransomware Group
- Attack Vector
- Ransom Amount (USD)

The target variable for this model is **Severity_Score**.

In [4]:
df.columns

Index(['Attack_ID', 'Company_Name', 'Industry', 'Country', 'Attack_Date',
       'Ransomware_Group', 'Attack_Vector', 'Ransom_Amount_USD',
       'Payment_Status', 'Data_Exfiltration', 'Downtime_Days',
       'Records_Affected', 'Estimated_Total_Loss_USD', 'Severity_Score'],
      dtype='str')

In [5]:
# Features selected for predicting Severity Score
selected_features = [
    "Industry",
    "Country",
    "Attack_Date",
    "Ransomware_Group",
    "Attack_Vector",
    "Ransom_Amount_USD"
]

In [6]:
X = df[selected_features]

# Target variable
y = df["Severity_Score"]

# Feature Types

Features are separated into numerical and categorical variables.

This allows different preprocessing techniques to be applied to each type.

In [7]:
categorical_features = X.select_dtypes(include="object").columns

numerical_features = X.select_dtypes(include=["int64","float64"]).columns

print(categorical_features)

print(numerical_features)

Index(['Industry', 'Country', 'Attack_Date', 'Ransomware_Group',
       'Attack_Vector'],
      dtype='str')
Index(['Ransom_Amount_USD'], dtype='str')


C:\Users\hlaku\AppData\Local\Temp\ipykernel_1472\3129469902.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns


In [8]:
X["Attack_Date"] = pd.to_datetime(X["Attack_Date"])

In [9]:
X["Attack_Year"] = X["Attack_Date"].dt.year

X["Attack_Month"] = X["Attack_Date"].dt.month

X["Attack_Day"] = X["Attack_Date"].dt.day

X["Attack_DayOfWeek"] = X["Attack_Date"].dt.dayofweek

In [10]:
categorical_features = X.select_dtypes(include="object").columns

numerical_features = X.select_dtypes(include=["int64","float64"]).columns

C:\Users\hlaku\AppData\Local\Temp\ipykernel_1472\701655839.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns


# Train-Test Split

The dataset is divided into training and testing subsets.

The training data is used to build the model, while the testing data evaluates its performance on unseen observations.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Preprocessing Pipeline

A preprocessing pipeline is created to ensure consistent transformation of both training and future prediction data.

Categorical variables are one-hot encoded.

Numerical variables are standardized.

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [14]:
print(X_train_processed.shape)

print(X_test_processed.shape)

(4000, 37)
(1000, 37)


In [15]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']

In [17]:
joblib.dump(X_train_processed, "../models/X_train.pkl")
joblib.dump(X_test_processed, "../models/X_test.pkl")

joblib.dump(y_train, "../models/y_train.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']

# Conclusion

The ransomware dataset has been successfully prepared for machine learning.

The following preprocessing tasks were completed:

- Removed unnecessary features
- Engineered new date-based features
- Separated numerical and categorical variables
- Split the dataset into training and testing sets
- Encoded categorical features
- Standardized numerical variables
- Saved the preprocessing pipeline

The processed datasets are now ready for model training in Notebook 4.